# Vyuha P11b — AgentDojo, all 4 suites, gpt-oss-120b via DeepInfra (paid, capped)

**CPU is enough, so no GPU quota is used.** Kaggle: Accelerator **None**, Internet **ON**, **Persistence → Files** (keeps the task cache across restarts), and secrets `AGENT_API_KEY` (your DeepInfra key) + `HF_TOKEN`.
Run cells **in order**. Cell 2 costs ≈$0 and Cell 3 a few cents. Cell 3 prints a projected full cost; only then run Cell 4, which has a hard spend cap.
Every finished user-task is cached, so re-running a cell resumes; it never re-spends money on finished work.

In [ ]:
# Cell 1 — setup (Kaggle or Colab)
import os, sys, glob, subprocess, json
KAGGLE = os.path.isdir('/kaggle/working')
WORK = '/kaggle/working' if KAGGLE else '/content'
DEST = f'{WORK}/vyuha_src'
if os.path.isdir(f'{DEST}/.git'):
    subprocess.run(['git', '-C', DEST, 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/g25ait2149/vyuha.git', DEST], check=False)
hits = glob.glob(DEST + '/**/vyuha/__init__.py', recursive=True)
assert hits, 'clone failed - turn Internet ON'
ROOT = os.path.dirname(os.path.dirname(hits[0])); sys.path.insert(0, ROOT); os.chdir(ROOT)
for m in [m for m in list(sys.modules) if m == 'vyuha' or m.startswith(('vyuha.', 'eval'))]:
    del sys.modules[m]
subprocess.run('pip -q install agentdojo openai 2>/dev/null', shell=True)
def _secret(name):
    try:
        if KAGGLE:
            from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret(name)
        from google.colab import userdata; return userdata.get(name)
    except Exception:
        return os.environ.get(name, '')
API_KEY = _secret('AGENT_API_KEY')
assert API_KEY, 'add your DeepInfra key as secret AGENT_API_KEY (attached to this notebook)'
LOGDIR = f'{WORK}/agentdojo_deepinfra'
MODEL = 'openai/gpt-oss-120b'
PRICE_IN, PRICE_OUT = 0.04, 0.17     # USD per 1M tokens - CHECK DeepInfra's model page and edit if different
print('platform:', 'Kaggle' if KAGGLE else 'Colab', '| key set | cache:', LOGDIR)


In [ ]:
# Cell 2 — smoke test (~$0): model exists + tool calling works on DeepInfra
import openai
cl = openai.OpenAI(api_key=API_KEY, base_url='https://api.deepinfra.com/v1/openai')
ids = [m.id for m in cl.models.list()]
print('gpt-oss models on DeepInfra:', [i for i in ids if 'gpt-oss' in i])
assert MODEL in ids, f'{MODEL} not listed - tell me which gpt-oss id appears above'
tools = [{'type': 'function', 'function': {'name': 'get_balance', 'description': 'Get the account balance',
          'parameters': {'type': 'object', 'properties': {'account': {'type': 'string'}}, 'required': ['account']}}}]
r = cl.chat.completions.create(model=MODEL, messages=[{'role': 'user', 'content': 'What is the balance of account 42? Use the tool.'}],
                               tools=tools, max_tokens=300)
tc = r.choices[0].message.tool_calls
print('tool call:', tc[0].function.name if tc else None, tc[0].function.arguments if tc else '')
print('usage:', r.usage.prompt_tokens, 'in /', r.usage.completion_tokens, 'out')
assert tc, 'model answered without calling the tool - send me this output before spending more'
print('SMOKE TEST OK')


In [ ]:
# Cell 3 — pilot (a few cents): 2 banking user-tasks x 2 injections, both arms; projects the full-run cost
from eval import agentdojo_eval as AD
AD.USAGE.update(prompt=0, completion=0, calls=0, usd=0.0)
pilot = AD.run_agentdojo_l3(api_key=API_KEY, provider='deepinfra', model=MODEL, suite_name='banking',
                            n_user_tasks=2, n_injection_tasks=2, rpm_interval=0.0, logdir=LOGDIR + '_pilot',
                            budget_usd=0.50, price_in=PRICE_IN, price_out=PRICE_OUT)
U = dict(AD.USAGE); episodes = sum(v['n'] for k, v in pilot.items() if isinstance(v, dict) and 'n' in v)
from agentdojo.task_suite.load_suites import get_suite
FULL_EPISODES = 2 * sum(len(get_suite('v1.2.2', x).user_tasks) * len(get_suite('v1.2.2', x).injection_tasks)
                        for x in ('banking', 'slack', 'travel', 'workspace'))   # = 1898 for v1.2.2
per_ep = U['usd'] / max(episodes, 1)
print(f"\npilot: {episodes} episodes, {U['calls']} calls, {U['prompt']:,} in / {U['completion']:,} out tokens, est ${U['usd']:.3f}")
print(f"projected FULL run: ~{FULL_EPISODES} episodes x ${per_ep:.4f} = ~${per_ep * FULL_EPISODES:.2f} (estimate; DeepInfra dashboard is the truth)")


In [ ]:
# Cell 4 — FULL run: all user tasks x all injection tasks, 4 suites, both arms. Hard spend cap.
BUDGET_USD = 8.00          # stop cleanly at this estimated spend; raise only if Cell 3's projection says so
SUITES = ('banking', 'slack', 'travel', 'workspace')
from eval import agentdojo_eval as AD
AD.USAGE.update(prompt=0, completion=0, calls=0, usd=0.0)
pooled = AD.run_agentdojo_l3_suites(api_key=API_KEY, provider='deepinfra', model=MODEL, suites=SUITES,
                                    n_user_tasks=None, n_injection_tasks=None, rpm_interval=0.0, logdir=LOGDIR,
                                    budget_usd=BUDGET_USD, price_in=PRICE_IN, price_out=PRICE_OUT)
json.dump({k: v for k, v in pooled.items()}, open(f'{LOGDIR}/pooled_results.json', 'w'), indent=1, default=str)
print(f"\nthis pass: {AD.USAGE['calls']} calls, {AD.USAGE['prompt']:,} in / {AD.USAGE['completion']:,} out, est ${AD.USAGE['usd']:.2f}")


In [ ]:
# Cell 5 — results table with Wilson 95% CIs (per suite + pooled). Send me this output.
from eval.metrics import wilson_ci
def row(name, d):
    n = d['n']; k_asr = round(d['injection_asr'] * n); k_u = round(d['utility_under_attack'] * n)
    a = wilson_ci(k_asr, n) if n else (0, 0); u = wilson_ci(k_u, n) if n else (0, 0)
    return (f"{name:<22}{d['injection_asr']:>6.2f} [{a[0]:.2f},{a[1]:.2f}]{d['utility_under_attack']:>8.2f} [{u[0]:.2f},{u[1]:.2f}]"
            f"{n:>6}{d.get('skipped', 0):>8}")
print(f"{'suite / arm':<22}{'inj. ASR [95% CI]':>20}{'utility-under-attack':>22}{'n':>6}{'skipped':>8}")
for s, r in pooled['per_suite'].items():
    for arm in ('undefended', 'Vyuha L3'):
        if arm in r: print(row(f'{s} / {arm}', r[arm]))
for arm in ('undefended', 'Vyuha L3'):
    print(row(f'POOLED / {arm}', pooled[arm]))
